In [6]:
!pip install rapidfuzz

Defaulting to user installation because normal site-packages is not writeable


In [8]:
import pandas as pd
import re
import difflib
import tkinter as tk
from tkinter import filedialog

# Funcție grafică pentru a selecta fișierul direct din calculator
def select_file():
    root = tk.Tk()
    root.withdraw() # Ascunde fereastra principală goală a tkinter
    root.attributes('-topmost', True) # Forțează fereastra de selectare să apară deasupra
    file_path = filedialog.askopenfilename(
        title="Selectează fișierul presales_data_sample.csv",
        filetypes=[("CSV Files", "*.csv")]
    )
    return file_path

print("Aștept selectarea fișierului... (Verifică dacă a apărut fereastra pop-up în bara de Windows)")
cale_fisier = select_file()

if not cale_fisier:
    print("[EROARE] Nu ai selectat niciun fișier. Te rog să rulezi din nou celula.")
else:
    print(f"[OK] Am încărcat fișierul de la adresa: {cale_fisier}")
    
    # 1. Încărcarea datelor brute
    df = pd.read_csv(cale_fisier, encoding='utf-8-sig')
    df.columns = df.columns.str.strip()

    # 2. Funcție pentru curățarea termenilor legali (Zgomot de business)
    def clean_company_name(name):
        if pd.isna(name) or not isinstance(name, str):
            return ""
        name = name.lower()
        suffixes = r'\b(inc|incorporated|llc|ltd|limited|pvt|private|co|corp|corporation|gmbh|sa|sas|srl)\b'
        name = re.sub(suffixes, '', name)
        name = re.sub(r'[^\w\s]', '', name)
        return name.strip()

    # Funcție nativă pentru calcularea scorului de similaritate text
    def get_text_similarity(str1, str2):
        if not str1 or not str2:
            return 0
        return difflib.SequenceMatcher(None, str1, str2).ratio() * 100

    print("Inițializare procesare nativă pe cele {} de rânduri...".format(len(df)))

    best_matches = {}
    grouped = df.groupby('input_row_key')

    for row_key, group in grouped:
        input_name_raw = group['input_company_name'].iloc[0] if 'input_company_name' in group.columns else ""
        input_name_clean = clean_company_name(input_name_raw)
        input_country = str(group['input_main_country_code'].iloc[0]).lower().strip() if 'input_main_country_code' in group.columns else ""
        input_city = str(group['input_main_city'].iloc[0]).lower().strip() if 'input_main_city' in group.columns else ""
        
        highest_score = -1
        best_veridion_id = None
        
        for idx, row in group.iterrows():
            if pd.isna(row['veridion_id']):
                continue
                
            score = 0
            v_name_clean = clean_company_name(row['company_name'])
            v_legal_clean = clean_company_name(row['company_legal_names'])
            v_commercial_clean = clean_company_name(row['company_commercial_names'])
            v_country = str(row['main_country_code']).lower().strip()
            v_city = str(row['main_city']).lower().strip()
            v_domain = str(row['website_domain']).lower().strip()
            
            # --- FILTRUL 1: Validare Geografică ---
            if input_country == v_country and input_country != 'nan':
                score += 30
            else:
                score -= 20
                
            # --- FILTRUL 2: String Matching nativ ---
            score_name = max(
                get_text_similarity(input_name_clean, v_name_clean),
                get_text_similarity(input_name_clean, v_legal_clean),
                get_text_similarity(input_name_clean, v_commercial_clean)
            )
            score += (score_name * 0.5)
            
            # --- FILTRUL 3: Oraș ---
            if input_city != 'nan' and v_city != 'nan' and len(input_city) > 2:
                if input_city in v_city or v_city in input_city:
                    score += 15
                    
            # --- FILTRUL 4: Domeniu Web ---
            if v_domain != 'nan' and len(input_name_clean) > 2:
                first_word = input_name_clean.split()[0] if len(input_name_clean.split()) > 0 else ""
                if first_word and first_word in v_domain:
                    score += 10
            
            if score > highest_score:
                highest_score = score
                best_veridion_id = row['veridion_id']
                
        # Prag de siguranță ajustat pentru algoritmul difflib
        if highest_score >= 50: 
            best_matches[row_key] = {
                'veridion_id': best_veridion_id,
                'status': 'Match choosen'
            }
        else:
            best_matches[row_key] = {
                'veridion_id': None,
                'status': 'To verify manually'
            }

    df['python_match_id'] = df['input_row_key'].map(lambda x: best_matches[x]['veridion_id'] if x in best_matches else None)
    df['python_status'] = df['input_row_key'].map(lambda x: best_matches[x]['status'] if x in best_matches else 'To verify manually')

    df['final_decision_python'] = df.apply(
        lambda r: 'Match choosen' if (r['veridion_id'] == r['python_match_id'] and pd.notna(r['veridion_id']) and r['python_status'] == 'Match choosen')
        else ('To verify manually' if r['python_status'] == 'To verify manually'
        else 'Excluded (have better match)'), axis=1
    )

    df.to_csv('presales_data_python_output.csv', index=False)
    print("\n[SUCCES] Procesare finalizată! Fișierul 'presales_data_python_output.csv' a fost generat.")

    print("\n--- DISTRIBUȚIA REZULTATELOR PYTHON ---")
    print(df['final_decision_python'].value_counts())

Aștept selectarea fișierului... (Verifică dacă a apărut fereastra pop-up în bara de Windows)
[OK] Am încărcat fișierul de la adresa: C:/Users/yomaj/Downloads/presales_data_sample - presales_data_sample.csv.csv
Inițializare procesare nativă pe cele 2951 de rânduri...

[SUCCES] Procesare finalizată! Fișierul 'presales_data_python_output.csv' a fost generat.

--- DISTRIBUȚIA REZULTATELOR PYTHON ---
final_decision_python
Excluded (have better match)    2245
Match choosen                    563
To verify manually               143
Name: count, dtype: int64


In [9]:
import os

# Găsim calea către folderul Downloads al utilizatorului tău
downloads_path = os.path.join(os.path.expanduser('~'), 'Downloads', 'presales_data_python_output.csv')

# Salvăm fișierul direct acolo
df.to_csv(downloads_path, index=False)
print(f"[SUCCES] Fișierul a fost mutat și salvat direct în Downloads:\n{downloads_path}")

[SUCCES] Fișierul a fost mutat și salvat direct în Downloads:
C:\Users\yomaj\Downloads\presales_data_python_output.csv
